# Instacart Rule-Based Evaluation

This notebook evaluates Apriori association rules as a recommender.

## Goals
- Load validation and test baskets
- Load recommendation rules mined from train_rules
- Predict hidden products from partial baskets
- Compute precision, recall, hit-rate, and coverage
- Compare results on validation and test_final
- Analyze results by customer segment

## Output
This notebook produces evaluation tables for the final project report.

In [8]:
import ast
from itertools import combinations
import pandas as pd
import numpy as np

In [9]:
def load_evaluation_inputs():
    """
    Load basket and rule files for evaluation.
    """
    tables = {
        "baskets_validation": pd.read_csv("../outputs/baskets_validation.csv"),
        "baskets_test": pd.read_csv("../outputs/baskets_test.csv"),
        "rules_reco": pd.read_csv("../outputs/apriori_train_rules_rules_reco.csv"),
    }
    return tables


def parse_items_column(baskets_df):
    """
    Convert the items column from string to list.
    """
    df = baskets_df.copy()
    df["items"] = df["items"].apply(ast.literal_eval)
    return df


def parse_rules_columns(rules_df):
    """
    Convert antecedents and consequents from text to sets.
    """
    df = rules_df.copy()

    def to_set(text):
        if pd.isna(text) or str(text).strip() == "":
            return set()
        return set(str(text).split("|"))

    df["antecedents_set"] = df["antecedents"].apply(to_set)
    df["consequents_set"] = df["consequents"].apply(to_set)
    return df


def basket_overview(baskets_df, name):
    """
    Build a simple basket summary table.
    """
    row = {
        "dataset": name,
        "n_baskets": len(baskets_df),
        "n_users": baskets_df["user_id"].nunique(),
        "avg_basket_size": baskets_df["items"].apply(len).mean(),
        "median_basket_size": baskets_df["items"].apply(len).median(),
    }
    return pd.DataFrame([row])


def rules_overview(rules_df):
    """
    Build a simple rules summary table.
    """
    if len(rules_df) == 0:
        return pd.DataFrame([{
            "n_rules": 0,
            "avg_confidence": np.nan,
            "avg_lift": np.nan,
            "avg_support": np.nan,
        }])

    row = {
        "n_rules": len(rules_df),
        "avg_confidence": rules_df["confidence"].mean(),
        "avg_lift": rules_df["lift"].mean(),
        "avg_support": rules_df["support"].mean(),
    }
    return pd.DataFrame([row])


def keep_valid_baskets_for_evaluation(baskets_df, min_items=2):
    """
    Keep baskets with at least min_items products.
    """
    df = baskets_df.copy()
    df["basket_size"] = df["items"].apply(len)
    df = df[df["basket_size"] >= min_items].copy()
    return df


def build_rules_index(rules_df):
    """
    Build a fast index: antecedent tuple -> list of rule rows.
    """
    rules_index = {}

    for _, row in rules_df.iterrows():
        consequents = list(row["consequents_set"])

        # Keep only subset -> 1 item rules
        if len(consequents) != 1:
            continue

        antecedent = tuple(sorted(list(row["antecedents_set"])))
        candidate = consequents[0]

        if antecedent not in rules_index:
            rules_index[antecedent] = []

        rules_index[antecedent].append({
            "candidate": candidate,
            "confidence": float(row["confidence"]),
            "lift": float(row["lift"]),
            "support": float(row["support"]),
        })

    return rules_index


def hide_items_in_basket(items, n_hide=1):
    """
    Hide the last n_hide items from a basket.
    """
    if len(items) <= n_hide:
        return None, None

    observed = items[:-n_hide]
    hidden = items[-n_hide:]

    if len(observed) == 0:
        return None, None

    return observed, hidden


def score_candidates_fast(observed_items, rules_index):
    """
    Score candidates using only matching antecedents from the basket.
    """
    observed = [str(x) for x in observed_items]
    observed_set = set(observed)

    # Build only useful antecedents from the observed basket
    # We use size 1 and size 2 because rules are mostly 1->1 and 2->1
    candidate_antecedents = []

    for item in observed:
        candidate_antecedents.append((item,))

    if len(observed) >= 2:
        for comb in combinations(sorted(observed), 2):
            candidate_antecedents.append(comb)

    matched_rules_count = 0
    scores = {}

    for ant in candidate_antecedents:
        if ant not in rules_index:
            continue

        for rule_info in rules_index[ant]:
            matched_rules_count += 1
            candidate = rule_info["candidate"]

            # Do not recommend an item already in the observed basket
            if candidate in observed_set:
                continue

            if candidate not in scores:
                scores[candidate] = {
                    "score": 0.0,
                    "max_confidence": 0.0,
                    "max_lift": 0.0,
                    "max_support": 0.0,
                }

            conf = rule_info["confidence"]
            lift = rule_info["lift"]
            supp = rule_info["support"]

            # Simple score: sum of confidence from matching rules
            scores[candidate]["score"] += conf
            scores[candidate]["max_confidence"] = max(
                scores[candidate]["max_confidence"], conf
            )
            scores[candidate]["max_lift"] = max(scores[candidate]["max_lift"], lift)
            scores[candidate]["max_support"] = max(
                scores[candidate]["max_support"], supp
            )

    rows = []
    for candidate, vals in scores.items():
        rows.append({
            "candidate": candidate,
            "score": vals["score"],
            "max_confidence": vals["max_confidence"],
            "max_lift": vals["max_lift"],
            "max_support": vals["max_support"],
        })

    if len(rows) == 0:
        return pd.DataFrame(), matched_rules_count

    out = pd.DataFrame(rows)
    out = out.sort_values(
        ["score", "max_lift", "max_support"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return out, matched_rules_count


def evaluate_baskets_with_hidden_item_fast(
    baskets_df,
    rules_df,
    top_k=10,
    n_hide=1,
    max_baskets=None
):
    """
    Evaluate rules by hiding items in baskets (fast version).
    """
    # Build the rule index once (faster than scanning all rules every time)
    rules_index = build_rules_index(rules_df)

    # Optional limit for quick tests
    df = baskets_df.copy()
    if max_baskets is not None:
        df = df.head(max_baskets).copy()

    evaluated_rows = []
    skipped = 0
    covered_baskets = 0

    for _, row in df.iterrows():
        items = row["items"]

        # Skip baskets that are too small
        observed, hidden = hide_items_in_basket(items, n_hide=n_hide)
        if observed is None:
            skipped += 1
            continue

        # Predict candidates from rules
        candidates_df, matched_rules_count = score_candidates_fast(
            observed,
            rules_index
        )

        hidden_set = set(str(x) for x in hidden)

        if len(candidates_df) == 0:
            preds = []
        else:
            preds = candidates_df["candidate"].head(top_k).tolist()

        pred_set = set(preds)

        if len(preds) > 0:
            covered_baskets += 1

        hits = len(pred_set.intersection(hidden_set))

        evaluated_rows.append({
            "order_id": row["order_id"],
            "user_id": row["user_id"],
            "segment": row["segment"] if "segment" in row.index else np.nan,
            "basket_size": len(items),
            "observed_size": len(observed),
            "hidden_size": len(hidden),
            "n_predictions": len(preds),
            "n_hits": hits,
            "precision_at_k": hits / top_k if top_k > 0 else 0.0,
            "recall_at_k": hits / len(hidden_set) if len(hidden_set) > 0 else 0.0,
            "hit_rate": 1.0 if hits > 0 else 0.0,
            "matched_rules_count": matched_rules_count,
        })

    eval_df = pd.DataFrame(evaluated_rows)

    if len(eval_df) == 0:
        summary = pd.DataFrame([{
            "n_evaluated_baskets": 0,
            "n_skipped_baskets": skipped,
            "top_k": top_k,
            "n_hide": n_hide,
            "mean_precision_at_k": np.nan,
            "mean_recall_at_k": np.nan,
            "hit_rate_at_k": np.nan,
            "coverage_at_k": np.nan,
        }])
        return eval_df, summary

    summary = pd.DataFrame([{
        "n_evaluated_baskets": len(eval_df),
        "n_skipped_baskets": skipped,
        "top_k": top_k,
        "n_hide": n_hide,
        "mean_precision_at_k": eval_df["precision_at_k"].mean(),
        "mean_recall_at_k": eval_df["recall_at_k"].mean(),
        "hit_rate_at_k": eval_df["hit_rate"].mean(),
        "coverage_at_k": covered_baskets / len(eval_df),
    }])

    return eval_df, summary


def evaluate_by_segment(eval_df):
    """
    Build evaluation summary by customer segment.
    """
    if len(eval_df) == 0:
        return pd.DataFrame()

    if "segment" not in eval_df.columns:
        return pd.DataFrame()

    out = eval_df.groupby("segment").agg(
        n_baskets=("order_id", "count"),
        mean_precision_at_k=("precision_at_k", "mean"),
        mean_recall_at_k=("recall_at_k", "mean"),
        hit_rate_at_k=("hit_rate", "mean"),
    ).reset_index()

    # Keep a fixed order for display
    segment_order = {"rare": 0, "frequent": 1, "heavy": 2}
    out["segment_order"] = out["segment"].map(segment_order).fillna(999)
    out = out.sort_values("segment_order").drop(columns="segment_order")
    out = out.reset_index(drop=True)

    return out


def compare_validation_test(summary_val, summary_test):
    """
    Combine validation and test summaries in one table.
    """
    val = summary_val.copy()
    val["dataset"] = "validation"

    test = summary_test.copy()
    test["dataset"] = "test_final"

    out = pd.concat([val, test], ignore_index=True)
    return out


def save_evaluation_outputs(eval_val, eval_test, summary_compare, seg_val, seg_test):
    """
    Save evaluation tables to the outputs folder.
    """
    eval_val.to_csv("../outputs/eval_validation_details.csv", index=False)
    eval_test.to_csv("../outputs/eval_test_final_details.csv", index=False)
    summary_compare.to_csv("../outputs/eval_summary_validation_test.csv", index=False)

    if len(seg_val) > 0:
        seg_val.to_csv("../outputs/eval_validation_by_segment.csv", index=False)

    if len(seg_test) > 0:
        seg_test.to_csv("../outputs/eval_test_final_by_segment.csv", index=False)

In [10]:
# Load files
tables = load_evaluation_inputs()

baskets_validation = parse_items_column(tables["baskets_validation"])
baskets_test = parse_items_column(tables["baskets_test"])
rules_reco = parse_rules_columns(tables["rules_reco"])

# Show input summaries
display(basket_overview(baskets_validation, "validation"))
display(basket_overview(baskets_test, "test_final"))
display(rules_overview(rules_reco))

,dataset,n_baskets,n_users,avg_basket_size,median_basket_size
0,validation,206209,206209,10.376792,9.0


,dataset,n_baskets,n_users,avg_basket_size,median_basket_size
0,test_final,131209,131209,10.552759,9.0


,n_rules,avg_confidence,avg_lift,avg_support
0,209,0.202982,2.137759,0.007138


In [11]:
# Keep baskets with at least 2 items
baskets_validation_eval = keep_valid_baskets_for_evaluation(
    baskets_validation,
    min_items=2
)
baskets_test_eval = keep_valid_baskets_for_evaluation(
    baskets_test,
    min_items=2
)

display(basket_overview(baskets_validation_eval, "validation_eval_ready"))
display(basket_overview(baskets_test_eval, "test_eval_ready"))

,dataset,n_baskets,n_users,avg_basket_size,median_basket_size
0,validation_eval_ready,195430,195430,10.893972,9.0


,dataset,n_baskets,n_users,avg_basket_size,median_basket_size
0,test_eval_ready,124364,124364,11.078544,9.0


In [12]:
# Evaluation parameters
top_k_value = 10
n_hide_value = 1

eval_params = pd.DataFrame([{
    "top_k": top_k_value,
    "n_hide": n_hide_value,
    "rule_logic": "subset_to_one_item",
}])
display(eval_params)


,top_k,n_hide,rule_logic
0,10,1,subset_to_one_item


In [13]:
# Evaluate on validation
eval_validation_details, eval_validation_summary = (
    evaluate_baskets_with_hidden_item_fast(
        baskets_validation_eval,
        rules_reco,
        top_k=top_k_value,
        n_hide=n_hide_value,
        max_baskets=None
    )
)
display(eval_validation_summary)

,n_evaluated_baskets,n_skipped_baskets,top_k,n_hide,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k,coverage_at_k
0,195430,0,10,1,0.002637,0.026373,0.026373,0.646564


In [14]:
# Evaluate on test_final
eval_test_details, eval_test_summary = (
    evaluate_baskets_with_hidden_item_fast(
        baskets_test_eval,
        rules_reco,
        top_k=top_k_value,
        n_hide=n_hide_value,
        max_baskets=None
    )
)
display(eval_test_summary)

,n_evaluated_baskets,n_skipped_baskets,top_k,n_hide,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k,coverage_at_k
0,124364,0,10,1,0.002635,0.02635,0.02635,0.650848


In [15]:
# Compare validation and test_final
eval_compare = compare_validation_test(
    eval_validation_summary,
    eval_test_summary
)
display(eval_compare)

,n_evaluated_baskets,n_skipped_baskets,top_k,n_hide,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k,coverage_at_k,dataset
0,195430,0,10,1,0.002637,0.026373,0.026373,0.646564,validation
1,124364,0,10,1,0.002635,0.026350,0.026350,0.650848,test_final


In [16]:
# Evaluation by segment
eval_validation_by_segment = evaluate_by_segment(eval_validation_details)
eval_test_by_segment = evaluate_by_segment(eval_test_details)

display(eval_validation_by_segment)
display(eval_test_by_segment)

,segment,n_baskets,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k
0,rare,60325,0.002105,0.021053,0.021053
1,frequent,66320,0.002557,0.025573,0.025573
2,heavy,68785,0.003181,0.031809,0.031809


,segment,n_baskets,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k
0,rare,38550,0.002086,0.020856,0.020856
1,frequent,41994,0.002596,0.025956,0.025956
2,heavy,43820,0.003156,0.031561,0.031561


In [17]:
# Show samples of detailed evaluation rows
display(eval_validation_details.head(20))
display(eval_test_details.head(20))

,order_id,user_id,segment,basket_size,observed_size,hidden_size,n_predictions,n_hits,precision_at_k,recall_at_k,hit_rate,matched_rules_count
0,2550362,1,frequent,9,8,1,1,0,0.0,0.0,0.0,1
1,839880,2,heavy,16,15,1,5,0,0.0,0.0,0.0,11
2,1402502,3,frequent,6,5,1,6,0,0.0,0.0,0.0,11
3,2557754,4,rare,3,2,1,0,0,0.0,0.0,0.0,0
4,157374,5,rare,12,11,1,6,0,0.0,0.0,0.0,10
5,998866,6,rare,3,2,1,0,0,0.0,0.0,0.0,0
6,2452257,7,heavy,12,11,1,5,0,0.0,0.0,0.0,10
7,2570360,8,rare,13,12,1,3,0,0.0,0.0,0.0,3
8,1830137,9,frequent,35,34,1,2,0,0.0,0.0,0.0,2
9,1353310,10,heavy,29,28,1,7,0,0.0,0.0,0.0,18


,order_id,user_id,segment,basket_size,observed_size,hidden_size,n_predictions,n_hits,precision_at_k,recall_at_k,hit_rate,matched_rules_count
0,1187899,1,frequent,11,10,1,5,0,0.0,0.0,0.0,7
1,1492625,2,heavy,31,30,1,4,0,0.0,0.0,0.0,10
2,2196797,5,rare,9,8,1,6,0,0.0,0.0,0.0,11
3,525192,7,heavy,9,8,1,0,0,0.0,0.0,0.0,0
4,880375,8,rare,18,17,1,6,0,0.0,0.0,0.0,17
5,1094988,9,frequent,22,21,1,2,0,0.0,0.0,0.0,2
6,1822501,10,heavy,4,3,1,0,0,0.0,0.0,0.0,0
7,1827621,13,frequent,5,4,1,1,0,0.0,0.0,0.0,1
8,2316178,14,heavy,11,10,1,0,0,0.0,0.0,0.0,0
9,2180313,17,heavy,6,5,1,2,0,0.0,0.0,0.0,3


In [18]:
# Save outputs
save_evaluation_outputs(
    eval_val=eval_validation_details,
    eval_test=eval_test_details,
    summary_compare=eval_compare,
    seg_val=eval_validation_by_segment,
    seg_test=eval_test_by_segment,
)

# notebook summary
final_summary = pd.DataFrame([{
    "n_validation_evaluated": len(eval_validation_details),
    "n_test_evaluated": len(eval_test_details),
    "validation_hit_rate_at_k": eval_validation_summary["hit_rate_at_k"].iloc[0],
    "test_hit_rate_at_k": eval_test_summary["hit_rate_at_k"].iloc[0],
    "outputs_saved": True,
}])
display(final_summary)

,n_validation_evaluated,n_test_evaluated,validation_hit_rate_at_k,test_hit_rate_at_k,outputs_saved
0,195430,124364,0.026373,0.02635,True
